# Reproducible evaluation of regional river ice models

This notebook produced daily ice-state evaluation, independent break-up/freeze-up timing, RF feature-set comparisons, and matched site-level trend analysis. 

**Two execution modes:** `reuse_predictions` regenerates the paper's tables, confidence intervals, and Figures 3–4 from archived CSVs without fitting models. `refit` additionally prepares predictors and repeats the nested model evaluation. Run cells in order from a fresh kernel. Full refitting is computationally intensive (35 outer splits × six RF specifications × 24 candidates × four inner folds, subject to available groups).

## 1 Configuration and software

Place the three documented input CSVs in `data/` for refitting, or place the archived prediction CSVs in `output/` for replay. All paths are relative and explicit. Source data are never overwritten; new outputs go to `publication_release_outputs/`.

The model run was run using Python 3.13.11, NumPy 2.2.2, pandas 2.3.3, and scikit-learn 1.9.0. Exact runtime versions and input hashes are exported for each execution.

In [ ]:
from pathlib import Path
import json
import platform
import warnings

import numpy as np
import pandas as pd

import sklearn
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, ParameterSampler, GridSearchCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss, log_loss,
)

SEED = 42
N_INNER_SPLITS = 4
N_OUTER_SPLITS = 5
N_SEARCH_ITER = 24
N_BOOT = 2000

# Switch to "refit" only when all documented source inputs are available.
MODE = "reuse_predictions"
DATA_DIR = Path("data")
CACHE_DIR = Path("output")
OUTPUT_DIR = Path("publication_release_outputs")
CLIMATE_PATH = DATA_DIR / "AK_Climate.csv"
OBS_PATH = DATA_DIR / "river_ice_validation.csv"
EVENT_PATH = DATA_DIR / "arp_freezeup_breakup_updated.csv"
LOCAL_EVENT_PATH = None
EVENT_PERSISTENCE_DAYS = 1  
PERSISTENCE_VALUES = (1, 3, 5)
TREND_SUMMARY = "median"  # Both mean and median outputs are labeled and exported.
TREND_N_BOOT = 10000
MIN_TREND_YEARS = 5
TREND_ALPHA = 0.05
RECORD_THRESHOLDS = (5, 8, 10, 15, 20)
FIGURE4_LIMITS = (-20, 5)  # Set None to show all sites; clipped counts are annotated.
SAVE_FITTED_MODELS = False  # Optional; can require substantial storage.
N_JOBS = -1

# ATDD correction used in this notebook:
#   ATDD = cumulative positive degree days from Jan 1 through Aug 31
#   ATDD = 0 from Sep 1 through Dec 31
RF_FEATURES = ["Tair", "vp", "SW", "LW", "ATDD", "AFDD", "rolling_avg_30", "MOY_sin", "MOY_cos"]
NO_CALENDAR_RF_FEATURES = ["Tair", "vp", "SW", "LW", "ATDD", "AFDD", "rolling_avg_30"]
NO_ATDD_RF_FEATURES = ["Tair", "vp", "SW", "LW", "AFDD", "rolling_avg_30", "MOY_sin", "MOY_cos"]
NO_AFDD_RF_FEATURES = ["Tair", "vp", "SW", "LW", "ATDD", "rolling_avg_30", "MOY_sin", "MOY_cos"]
TEMP_RF_FEATURES = ["rolling_avg_30"]
TEMP_CALENDAR_RF_FEATURES = ["rolling_avg_30", "MOY_sin", "MOY_cos"]

MODEL_FEATURE_SETS = {
    "Random forest": RF_FEATURES,
    "RF without calendar": NO_CALENDAR_RF_FEATURES,
    "RF without ATDD": NO_ATDD_RF_FEATURES,
    "RF without AFDD": NO_AFDD_RF_FEATURES,
    "Temperature-only RF": TEMP_RF_FEATURES,
    "Temperature + calendar RF": TEMP_CALENDAR_RF_FEATURES,
}

COMMON_EXCLUDED_MONTHS = [7, 8]
ISO_TEMP_COL = "rolling_avg_30"


import hashlib
import importlib.metadata
import sys
import zlib
from itertools import combinations
import scipy
from scipy.stats import norm, theilslopes
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
from IPython.display import display

if MODE not in {"reuse_predictions", "refit"}:
    raise ValueError("MODE must be reuse_predictions or refit.")
if TREND_SUMMARY not in {"mean", "median"}:
    raise ValueError("TREND_SUMMARY must be mean or median.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_LABELS = {"Random forest": "Full RF", "Brown refit": "Temperature LR", "0 C isotherm": "0 °C isotherm"}
PRIMARY_MODELS = list(MODEL_LABELS)
TREND_SERIES = ["Observed", *PRIMARY_MODELS]
software_versions = {name: importlib.metadata.version(name) for name in
                     ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib"]}
software_versions["python"] = platform.python_version()
input_manifest = []

def record_input(path, role):
    path = Path(path).expanduser().resolve(strict=True)
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    input_manifest.append(dict(filename=path.name, role=role, bytes=path.stat().st_size,
                               sha256=digest.hexdigest()))
    return path

def save_csv(df, name):
    df.to_csv(OUTPUT_DIR / name, index=False)

display(pd.Series(software_versions, name="version").to_frame())
print("Mode:", MODE, "| primary persistence:", EVENT_PERSISTENCE_DAYS,
      "| trend summary:", TREND_SUMMARY)

## 2 Input schemas and preparation

Refitting requires:

| Input | Required columns | Provenance and units |
|---|---|---|
| `AK_Climate.csv` | `COMID`, `Date`, `Tair`, `vp`, `SW`, `LW` | Catchment-aggregated daily RASM forcing. `Tair` must be °C. |
| `river_ice_validation.csv` | `COMID`, `Date`, `is_ice` | Prepared daily labels, binary 0/1. |
| `arp_freezeup_breakup.csv` | `COMID`, `Date`, `Season`, `latitude`, `longitude` | Arp and Cherry event records after river-segment matching. `Season`: `break-up` or `freeze-up`. Winter year is calculated from the event date. |

The rolling 30-day temperature includes the current day, so this is a nowcast. AFDD starts on 1 September; ATDD accumulates January–August and is zero September–December. July and August are excluded from daily scoring. Duplicate daily labels are resolved to ice when any label indicates ice, matching the source workflow. Continuous forcing is required for accumulated degree days; missing days cannot be interpreted as zero temperature.

In [ ]:
def add_hydroclimate_features(clim):
    clim = clim.copy()
    clim["Date"] = pd.to_datetime(clim["Date"])

    dup = clim.duplicated(["COMID", "Date"], keep=False)
    if dup.any():
        raise ValueError(f"Climate table has {dup.sum()} duplicated COMID-Date rows; resolve before modeling.")

    required = {"COMID", "Date", "Tair", "vp", "SW", "LW"}
    if not required.issubset(clim):
        raise ValueError(f"Missing climate columns: {sorted(required - set(clim))}")
    if clim[list(required)].isna().any().any():
        raise ValueError("Missing climate values: resolve and document them before degree-day accumulation.")
    clim = clim.sort_values(["COMID", "Date"]).copy()
    gaps = clim.groupby("COMID")["Date"].diff().dropna()
    if not gaps.eq(pd.Timedelta(days=1)).all():
        raise ValueError("Climate records must contain consecutive daily dates within each COMID.")
    clim["Year"] = clim["Date"].dt.year
    clim["Month"] = clim["Date"].dt.month
    clim["Wt_year"] = np.where(clim["Month"] >= 9, clim["Year"] + 1, clim["Year"])
    clim["FDD"] = np.where(clim["Tair"] < 0, -clim["Tair"], 0.0)
    clim["TDD"] = np.where(clim["Tair"] > 0, clim["Tair"], 0.0)
    clim["AFDD"] = clim.groupby(["COMID", "Wt_year"], sort=False)["FDD"].cumsum()

    atdd_calendar = clim.groupby(["COMID", "Year"], sort=False)["TDD"].cumsum()
    clim["ATDD"] = np.where(clim["Month"] >= 9, 0.0, atdd_calendar)

    clim["MOY_sin"] = np.sin(2 * np.pi * clim["Month"] / 12)
    clim["MOY_cos"] = np.cos(2 * np.pi * clim["Month"] / 12)

    # Calendar-day rolling window; includes the prediction day's Tair.
    # If predictions must be issued before observing same-day Tair, shift Tair by 1 day first.
    rolling_parts = []
    for _, g in clim.groupby("COMID", sort=False):
        g = g.sort_values("Date").copy()
        g["rolling_avg_30"] = (
            g.set_index("Date")["Tair"]
            .rolling("30D", min_periods=30)
            .mean()
            .to_numpy()
        )
        rolling_parts.append(g)
    return pd.concat(rolling_parts, ignore_index=True)

def prepare_daily_predictors(climate_path=CLIMATE_PATH):
    clim = pd.read_csv(climate_path)
    daily = add_hydroclimate_features(clim)
    daily = daily[daily["Year"].between(1990, 2021)].copy()
    daily = daily[daily["Wt_year"] > 1991].copy()
    return daily.sort_values(["COMID", "Date"]).reset_index(drop=True)

def prepare_daily_ice_labels(obs_path=OBS_PATH):
    obs = pd.read_csv(obs_path)
    obs["Date"] = pd.to_datetime(obs["Date"])
    required = {"COMID", "Date", "is_ice"}
    missing = required - set(obs.columns)
    if missing:
        raise KeyError(f"Daily ice-label file is missing {sorted(missing)}.")
    labels = obs[["COMID", "Date", "is_ice"]].dropna(subset=["COMID", "Date", "is_ice"]).copy()
    if not labels["is_ice"].isin([0, 1]).all():
        raise ValueError("is_ice must contain only 0 or 1 before integer conversion.")
    labels["is_ice"] = labels["is_ice"].astype(int)
    if not set(labels["is_ice"].unique()).issubset({0, 1}):
        raise ValueError("is_ice must be binary (0/1).")

    n_conflict = (labels.groupby(["COMID", "Date"])["is_ice"].nunique() > 1).sum()
    print(f"Conflicting repeated COMID-days resolved as ice if any observation is ice: {n_conflict:,}")
    return (
        labels.groupby(["COMID", "Date"], as_index=False)["is_ice"]
        .max()
        .sort_values(["COMID", "Date"])
        .reset_index(drop=True)
    )

def attach_ice_labels(daily_predictors, daily_labels):
    feature_columns = sorted(set().union(*MODEL_FEATURE_SETS.values()))
    keep = ["COMID", "Date", "Year", "Month", "Wt_year"] + feature_columns
    missing = sorted(set(keep) - set(daily_predictors.columns))
    if missing:
        raise KeyError(f"Daily predictor table is missing {missing}.")

    df = daily_labels.merge(
        daily_predictors[keep],
        on=["COMID", "Date"],
        how="inner",
        validate="one_to_one",
    )
    required = ["COMID", "Date", "is_ice"] + feature_columns
    df = df.dropna(subset=required).copy()
    df = df[~df["Month"].isin(COMMON_EXCLUDED_MONTHS)].copy()
    df = df.sort_values(["COMID", "Date"]).reset_index(drop=True)
    assert not df.duplicated(["COMID", "Date"]).any()
    return df

## 3 Shared models and tuning

The spatial groups, 24 sampled candidates, inner folds, and random-forest seed are shared across feature sets within each outer split. Inner cross-validation groups river segments and minimizes Brier score. The deterministic isotherm uses `T30 <= 0`. Log loss for its hard 0/1 predictions is omitted from publication tables.

In [ ]:
def fit_brown_style(train):
    models = {}
    for season, months in {"freeze": [9, 10, 11, 12, 1, 2], "breakup": [3, 4, 5, 6]}.items():
        d = train[train["Month"].isin(months)].copy()
        if d["is_ice"].nunique() < 2:
            raise ValueError(f"{season}: training data contain only one outcome class.")
        model = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
        model.fit(d[["rolling_avg_30"]], d["is_ice"])
        models[season] = model
    return models


def predict_brown_style(models, df):
    probability = np.full(len(df), np.nan, dtype=float)
    for season, months in {"freeze": [9, 10, 11, 12, 1, 2], "breakup": [3, 4, 5, 6]}.items():
        mask = df["Month"].isin(months).to_numpy()
        if mask.any():
            probability[mask] = models[season].predict_proba(df.loc[mask, ["rolling_avg_30"]])[:, 1]
    return probability


def isotherm_probability(df, temp_col=ISO_TEMP_COL):
    return (df[temp_col].to_numpy(dtype=float) <= 0.0).astype(float)


rf_param_distributions = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [None, 5, 10, 20, 40],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_features": ["sqrt", "log2", 0.5, 1.0],
    "bootstrap": [True, False],
    "criterion": ["gini", "entropy"],
}


# Publication protocol controls.  These should be fixed before inspecting the final results.
PROTOCOL_N_INNER_SPLITS = N_INNER_SPLITS
PROTOCOL_N_SEARCH_ITER = N_SEARCH_ITER
PROTOCOL_N_BOOT = N_BOOT
PROTOCOL_THRESHOLD = 0.50
PROTOCOL_SPATIAL_SPLITS = N_OUTER_SPLITS

# Predeclare forecasting periods.  These are intentionally identical across the
# temporal and space/time analyses.
FORECAST_ORIGINS = [
    dict(name="1990_1999_to_2000_2005", train_end=1999, test_start=2000, test_end=2005),
    dict(name="1990_2005_to_2006_2010", train_end=2005, test_start=2006, test_end=2010),
    dict(name="1990_2010_to_2011_2015", train_end=2010, test_start=2011, test_end=2015),
    dict(name="1990_2015_to_2016_2021", train_end=2015, test_start=2016, test_end=2021),
]

RF_MODEL_PROB_COLS = {
    "Random forest": "rf_prob",
    "RF without calendar": "rf_no_calendar_prob",
    "RF without ATDD": "rf_no_atdd_prob",
    "RF without AFDD": "rf_no_afdd_prob",
    "Temperature-only RF": "temp_rf_prob",
    "Temperature + calendar RF": "temp_calendar_rf_prob",
}

MODEL_SPECS = {
    **RF_MODEL_PROB_COLS,
    "Brown refit": "brown_refit_prob",
    "0 C isotherm": "isotherm_prob",
}

TEMPERATURE_BASED_MODELS = [
    "Temperature-only RF",
    "Temperature + calendar RF",
    "Brown refit",
    "0 C isotherm",
]

FREEZEUP_ABLATION_MODELS = [
    "Random forest",
    "RF without calendar",
    "RF without ATDD",
    "RF without AFDD",
    "Temperature-only RF",
    "Temperature + calendar RF",
]


def fit_models_for_protocol(train, seed):
    feature_columns = sorted(set().union(*MODEL_FEATURE_SETS.values()))
    required = feature_columns + ["is_ice", "COMID", "Month"]
    train = train.dropna(subset=required).copy()
    if train.COMID.nunique() < 2 or train.is_ice.nunique() < 2:
        raise ValueError("A protocol training set needs at least two COMIDs and both ice classes.")
    n_inner = min(PROTOCOL_N_INNER_SPLITS, train.COMID.nunique())
    inner_cv = GroupKFold(n_splits=n_inner)
    base_rf = RandomForestClassifier(random_state=seed, n_jobs=1, class_weight=None)
    
    candidate_params = list(ParameterSampler(
        rf_param_distributions,
        n_iter=PROTOCOL_N_SEARCH_ITER,
        random_state=seed,
    ))
    
    # GridSearchCV needs each parameter value wrapped in a list.
    candidate_param_grid = [
        {key: [value] for key, value in params.items()}
        for params in candidate_params
    ]
    
    def tune(features):
        search = GridSearchCV(
            estimator=clone(base_rf),
            param_grid=candidate_param_grid,
            scoring="neg_brier_score",
            cv=inner_cv,
            n_jobs=N_JOBS,
            refit=True,
            error_score="raise",
        )
        search.fit(train[features], train.is_ice, groups=train.COMID)
        return search
    
    rf_searches = {}
    for model_name, features in MODEL_FEATURE_SETS.items():
        rf_searches[model_name] = tune(features)

    brown = fit_brown_style(train)
    fitted = {
        "rf_models": {model: search.best_estimator_ for model, search in rf_searches.items()},
        "rf_params": {model: search.best_params_ for model, search in rf_searches.items()},
        "rf_inner_brier": {model: -search.best_score_ for model, search in rf_searches.items()},
        "brown": brown,
    }
    return fitted

def predict_models_for_protocol(df, fitted):
    out = df.copy()
    for model_name, features in MODEL_FEATURE_SETS.items():
        prob_col = RF_MODEL_PROB_COLS[model_name]
        out[prob_col] = fitted["rf_models"][model_name].predict_proba(out[features])[:, 1]
    out["brown_refit_prob"] = predict_brown_style(fitted["brown"], out)
    out["isotherm_prob"] = isotherm_probability(out)
    return out

def event_window(start_year, end_year):
    # Include the previous autumn so freeze-up for the first test winter is available.
    return pd.Timestamp(f"{start_year - 1}-09-01"), pd.Timestamp(f"{end_year}-06-30")

## 4 Event definitions and split construction

Break-up is the first ice-to-open-water transition during March–June. Freeze-up is the first open-water-to-ice transition during September–January. Persistence refers to consecutive **calendar days**, not merely consecutive rows. If duplicate source events map to the same segment/year/type, the source notebook's median-timestamp aggregation is retained. Source events without a valid transition remain missing; detection rates and complete-case attrition are reported.

In [ ]:
def load_reference_events(path=EVENT_PATH):
    path = Path(path)
    if not path.exists() and LOCAL_EVENT_PATH is not None and LOCAL_EVENT_PATH.exists():
        path = LOCAL_EVENT_PATH
    raw = pd.read_csv(path)
    required = {"latitude", "longitude", "Date", "Season", "COMID"}
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"Reference-event file is missing {sorted(missing)}.")
    d = raw.copy()
    d["observed_date"] = pd.to_datetime(d["Date"], errors="coerce")
    if d["observed_date"].isna().any():
        raise ValueError("At least one reference event date could not be parsed.")
    d["event_type"] = d["Season"].map({"break-up": "breakup", "freeze-up": "freezeup"})
    if d["event_type"].isna().any():
        raise ValueError("Unexpected Season value in the reference-event file.")
    month, year = d.observed_date.dt.month, d.observed_date.dt.year
    valid = ((d.event_type.eq("breakup") & month.isin([3, 4, 5, 6])) |
             (d.event_type.eq("freezeup") & month.isin([9, 10, 11, 12, 1])))
    d = d[valid].copy()
    month, year = d.observed_date.dt.month, d.observed_date.dt.year
    d["WinterYear"] = np.where(d.event_type.eq("freezeup") & month.ge(9), year + 1, year).astype(int)

    if "Wt_year" in d.columns:
        mismatch = d["Wt_year"].notna() & (pd.to_numeric(d["Wt_year"], errors="coerce") != d["WinterYear"])
        if mismatch.any():
            print(f"Event-file Wt_year differs from recomputed WinterYear for {mismatch.sum():,} rows; using recomputed WinterYear.")

    def median_timestamp(values):
        values = pd.Series(pd.to_datetime(values)).sort_values().astype("int64").to_numpy()
        return pd.to_datetime(int(np.median(values)))

    return (d.groupby(["COMID", "WinterYear", "event_type"], as_index=False)
             .agg(observed_date=("observed_date", median_timestamp),
                  n_source_events=("observed_date", "size"),
                  latitude=("latitude", "mean"), longitude=("longitude", "mean"))
             .sort_values(["COMID", "WinterYear", "event_type"]).reset_index(drop=True))

def first_transition_date(g, state_col, event_type, winter_year, persistence_days=EVENT_PERSISTENCE_DAYS):
    if event_type == "breakup":
        start, end, before, after = pd.Timestamp(f"{winter_year}-03-01"), pd.Timestamp(f"{winter_year}-06-30"), 1, 0
    else:
        start, end, before, after = pd.Timestamp(f"{winter_year - 1}-09-01"), pd.Timestamp(f"{winter_year}-01-31"), 0, 1
    w = g[g.Date.between(start, end)].sort_values("Date")
    if len(w) < 2:
        return pd.NaT
    if int(persistence_days) != persistence_days or persistence_days < 1:
        raise ValueError("persistence_days must be a positive integer.")
    state = w[state_col].to_numpy()
    dates = w.Date.reset_index(drop=True)
    idx = np.flatnonzero((state[:-1] == before) & (state[1:] == after))
    for i in idx:
        pos = i + 1
        end_pos = pos + int(persistence_days)
        consecutive = (dates.iloc[pos:min(end_pos, len(dates))].diff().dropna()
                       .eq(pd.Timedelta(days=1)).all())
        adjacent = dates.iloc[pos] - dates.iloc[pos - 1] == pd.Timedelta(days=1)
        if end_pos <= len(state) and adjacent and consecutive and np.all(state[pos:end_pos] == after):
            return pd.Timestamp(dates.iloc[pos])
    return pd.NaT

def extract_event_predictions(daily, refs, persistence_days=EVENT_PERSISTENCE_DAYS):
    daily = daily.copy()
    state_cols = {}
    for model, prob_col in MODEL_SPECS.items():
        state_col = prob_col.replace("_prob", "_state")
        daily[state_col] = np.where(daily[prob_col].notna(),
                                     (daily[prob_col] >= PROTOCOL_THRESHOLD).astype(float), np.nan)
        state_cols[model] = state_col
    groups = {comid: g for comid, g in daily.groupby("COMID")}
    rows = []
    for event in refs.itertuples(index=False):
        if event.COMID not in groups:
            continue
        row = {"COMID": event.COMID, "WinterYear": int(event.WinterYear),
               "event_type": event.event_type, "observed_date": pd.Timestamp(event.observed_date),
               "persistence_days": int(persistence_days)}
        for model, state_col in state_cols.items():
            row[model] = first_transition_date(
                groups[event.COMID], state_col, event.event_type, int(event.WinterYear), persistence_days=persistence_days
            )
        rows.append(row)
    return pd.DataFrame(rows)

def build_publication_splits(df):
    specs = []
    spatial_cv = GroupKFold(n_splits=PROTOCOL_SPATIAL_SPLITS)
    comid_table = pd.DataFrame({"COMID": np.sort(df["COMID"].unique())})
    spatial_id_splits = list(spatial_cv.split(comid_table, groups=comid_table["COMID"]))

    # Space: complete COMID trajectories are held out.
    for fold, (tr, te) in enumerate(spatial_id_splits, 1):
        train_ids = set(comid_table.iloc[tr]["COMID"])
        test_ids = set(comid_table.iloc[te]["COMID"])
        specs.append(dict(
            evaluation="space", split_id=f"space_{fold}", outer_fold=fold,
            train=df[df.COMID.isin(train_ids)].copy(),
            test=df[df.COMID.isin(test_ids)].copy(),
            train_end=np.nan, test_start=np.nan, test_end=np.nan,
            test_comids=test_ids,
        ))

    # Time: all locations remain available, but every test observation is later
    # than its corresponding training period.
    for origin in FORECAST_ORIGINS:
        specs.append(dict(
            evaluation="time", split_id=f"time_{origin['name']}", outer_fold=np.nan,
            train=df[df.Year.le(origin["train_end"])].copy(),
            test=df[df.Year.between(origin["test_start"], origin["test_end"])].copy(),
            train_end=origin["train_end"], test_start=origin["test_start"], test_end=origin["test_end"],
            test_comids=set(comid_table.COMID),
        ))

    # Space/time: future observations at completely unseen COMIDs.  The same
    # group folds and forecast origins are crossed, yielding 5 x 4 test sets.
    for origin in FORECAST_ORIGINS:
        for fold, (tr, te) in enumerate(spatial_id_splits, 1):
            train_ids = set(comid_table.iloc[tr]["COMID"])
            test_ids = set(comid_table.iloc[te]["COMID"])
            specs.append(dict(
                evaluation="space_time", split_id=f"space_time_{origin['name']}_fold_{fold}", outer_fold=fold,
                train=df[df.COMID.isin(train_ids) & df.Year.le(origin["train_end"])].copy(),
                test=df[df.COMID.isin(test_ids) & df.Year.between(origin["test_start"], origin["test_end"])].copy(),
                train_end=origin["train_end"], test_start=origin["test_start"], test_end=origin["test_end"],
                test_comids=test_ids,
            ))
    return specs

def build_fixed_hindcast_splits(df):
    specs = []
    all_comids = set(df["COMID"].unique())
    specs.append(dict(
        evaluation="fixed_time",
        split_id=f"fixed_train_to_{FIXED_HINDCAST_TRAIN_END}_test_{FIXED_HINDCAST_TEST_START}_{FIXED_HINDCAST_TEST_END}",
        outer_fold=np.nan,
        train=df[df.Year.le(FIXED_HINDCAST_TRAIN_END)].copy(),
        test=df[df.Year.between(FIXED_HINDCAST_TEST_START, FIXED_HINDCAST_TEST_END)].copy(),
        train_end=FIXED_HINDCAST_TRAIN_END,
        test_start=FIXED_HINDCAST_TEST_START,
        test_end=FIXED_HINDCAST_TEST_END,
        test_comids=all_comids,
    ))

    spatial_cv = GroupKFold(n_splits=PROTOCOL_SPATIAL_SPLITS)
    comid_table = pd.DataFrame({"COMID": np.sort(df["COMID"].unique())})
    for fold, (tr, te) in enumerate(spatial_cv.split(comid_table, groups=comid_table["COMID"]), 1):
        train_ids = set(comid_table.iloc[tr]["COMID"])
        test_ids = set(comid_table.iloc[te]["COMID"])
        specs.append(dict(
            evaluation="fixed_space_time",
            split_id=f"fixed_space_time_train_to_{FIXED_HINDCAST_TRAIN_END}_test_{FIXED_HINDCAST_TEST_START}_{FIXED_HINDCAST_TEST_END}_fold_{fold}",
            outer_fold=fold,
            train=df[df.COMID.isin(train_ids) & df.Year.le(FIXED_HINDCAST_TRAIN_END)].copy(),
            test=df[df.COMID.isin(test_ids) & df.Year.between(FIXED_HINDCAST_TEST_START, FIXED_HINDCAST_TEST_END)].copy(),
            train_end=FIXED_HINDCAST_TRAIN_END,
            test_start=FIXED_HINDCAST_TEST_START,
            test_end=FIXED_HINDCAST_TEST_END,
            test_comids=test_ids,
        ))
    return specs

FIXED_HINDCAST_TRAIN_END = 1999
FIXED_HINDCAST_TEST_START = 2000
FIXED_HINDCAST_TEST_END = 2021

def winter_season_day(date, winter_year):
    if pd.isna(date):
        return np.nan
    anchor = pd.Timestamp(year=int(winter_year) - 1, month=9, day=1)
    return (pd.Timestamp(date).normalize() - anchor).days

def make_event_trend_long(events):
    e = events.copy()
    date_cols = {"Observed": "observed_date"}
    date_cols.update({model: model for model in MODEL_SPECS})

    for col in date_cols.values():
        e[col] = pd.to_datetime(e[col], errors="coerce")

    rows = []
    for _, r in e.iterrows():
        for series, col in date_cols.items():
            if pd.isna(r[col]):
                continue
            rows.append({
                "evaluation": r["evaluation"],
                "split_id": r["split_id"],
                "COMID": r["COMID"],
                "WinterYear": int(r["WinterYear"]),
                "event_type": r["event_type"],
                "series": series,
                "date": r[col],
                "phenology_day": winter_season_day(r[col], r["WinterYear"]),
            })
    return pd.DataFrame(rows)

## 5 Generate or load the archived predictions

The replay mode requires these exact cached files: `daily_out_of_sample_predictions.csv`, `independent_event_predictions.csv`, `nested_tuning_by_protocol_split.csv`, `fixed_hindcast_tuning_by_split.csv`, and `fixed_hindcast_phenology_event_trend_long.csv`.

For replay, primary persistence is read from the event CSV and checked against the setting above. The refit mode preserves the original outer split order and seed formulas. Per-split daily predictions, event dates, tuning results, and exact inner COMID assignments are written as checkpoints. No existing files in the input/cache directories are changed.

In [ ]:
def read_cache(name, date_columns=()):
    path = record_input(CACHE_DIR / name, "archived prediction/tuning input")
    table = pd.read_csv(path)
    for column in date_columns:
        if column in table:
            table[column] = pd.to_datetime(table[column], errors="raise")
    return table

def fit_evaluation(specs, daily_clim, references, fixed=False):
    daily_parts, event_parts, window_parts, ref_parts, tuning_rows, membership = [], [], [], [], [], []
    checkpoint_dir = OUTPUT_DIR / "checkpoints"
    checkpoint_dir.mkdir(exist_ok=True)
    features = sorted(set().union(*MODEL_FEATURE_SETS.values()))
    for number, spec in enumerate(specs, 1):
        train, test = spec["train"], spec["test"]
        split_seed = SEED + (9000 if fixed else 0) + number * 100
        print(f"[{number}/{len(specs)}] {spec['split_id']} | seed={split_seed}")
        if set(train.COMID) & set(test.COMID) and spec["evaluation"] in {"space", "space_time", "fixed_space_time"}:
            raise ValueError("Spatial train/test COMIDs overlap.")
        fitted = fit_models_for_protocol(train, seed=split_seed)
        inner = list(GroupKFold(n_splits=min(N_INNER_SPLITS, train.COMID.nunique()))
                     .split(train, groups=train.COMID))
        for fold, (tr, te) in enumerate(inner, 1):
            for role, indices in [("train", tr), ("validation", te)]:
                membership.extend(dict(split_id=spec["split_id"], inner_fold=fold,
                                       role=role, COMID=c)
                                  for c in np.sort(train.iloc[indices].COMID.unique()))
        pred = predict_models_for_protocol(test, fitted)
        for col in ["evaluation", "split_id", "outer_fold"]:
            pred[col] = spec[col]
        daily_cols = ["COMID", "Date", "Year", "Month", "Wt_year", "is_ice",
                      "evaluation", "split_id", "outer_fold", *MODEL_SPECS.values()]
        daily_parts.append(pred[daily_cols])
        if pd.isna(spec["test_start"]):
            windows = daily_clim[daily_clim.COMID.isin(spec["test_comids"])].copy()
            refs = references[references.COMID.isin(spec["test_comids"]) &
                              references.WinterYear.between(daily_clim.Wt_year.min(), daily_clim.Wt_year.max())].copy()
        else:
            start, end = event_window(int(spec["test_start"]), int(spec["test_end"]))
            windows = daily_clim[daily_clim.COMID.isin(spec["test_comids"]) & daily_clim.Date.between(start, end)].copy()
            refs = references[references.COMID.isin(spec["test_comids"]) &
                              references.WinterYear.between(int(spec["test_start"]), int(spec["test_end"]))].copy()
        windows = predict_models_for_protocol(windows.dropna(subset=features), fitted)
        for col in ["evaluation", "split_id", "outer_fold"]:
            windows[col] = spec[col]
        refs["evaluation"], refs["split_id"] = spec["evaluation"], spec["split_id"]
        ev = extract_event_predictions(windows, refs, EVENT_PERSISTENCE_DAYS)
        ev["evaluation"], ev["split_id"] = spec["evaluation"], spec["split_id"]
        window_parts.append(windows[["COMID", "Date", "Year", "Month", "Wt_year",
                                     "evaluation", "split_id", "outer_fold", *MODEL_SPECS.values()]])
        event_parts.append(ev)
        ref_parts.append(refs)
        tuning_rows.append(dict(evaluation=spec["evaluation"], split_id=spec["split_id"],
            seed=split_seed, rf_best_params=json.dumps(fitted["rf_params"], sort_keys=True),
            rf_inner_brier=json.dumps(fitted["rf_inner_brier"], sort_keys=True)))
        pred[daily_cols].to_csv(checkpoint_dir / f"{spec['split_id']}_daily.csv", index=False)
        ev.to_csv(checkpoint_dir / f"{spec['split_id']}_events.csv", index=False)
        if SAVE_FITTED_MODELS:
            import joblib
            joblib.dump(fitted, checkpoint_dir / f"{spec['split_id']}_models.joblib")
    return (pd.concat(daily_parts, ignore_index=True), pd.concat(event_parts, ignore_index=True),
            pd.concat(window_parts, ignore_index=True), pd.concat(ref_parts, ignore_index=True),
            pd.DataFrame(tuning_rows), pd.DataFrame(membership))

if MODE == "reuse_predictions":
    protocol_daily_predictions = read_cache("daily_out_of_sample_predictions.csv", ["Date"])
    protocol_event_predictions = read_cache("independent_event_predictions.csv", ["observed_date", *MODEL_SPECS])
    protocol_tuning = read_cache("nested_tuning_by_protocol_split.csv")
    fixed_tuning = read_cache("fixed_hindcast_tuning_by_split.csv")
    fixed_event_trend_long = read_cache("fixed_hindcast_phenology_event_trend_long.csv", ["date"])
else:
    for path, role in [(CLIMATE_PATH, "daily RASM forcing"), (OBS_PATH, "prepared daily labels"),
                       (EVENT_PATH, "river-matched event observations")]:
        record_input(path, role)
    daily_clim = prepare_daily_predictors()
    labels = prepare_daily_ice_labels()
    Obs_df = attach_ice_labels(daily_clim, labels)
    reference_events = load_reference_events()
    save_csv(reference_events[["COMID", "latitude", "longitude"]].drop_duplicates(), "observation_locations.csv")
    publication_splits = build_publication_splits(Obs_df)
    fixed_splits = build_fixed_hindcast_splits(Obs_df)
    split_audit = pd.DataFrame([
        dict(evaluation=s["evaluation"], split_id=s["split_id"], seed=SEED + offset + i * 100,
             train_end=s["train_end"], test_start=s["test_start"], test_end=s["test_end"],
             n_train=len(s["train"]), n_test=len(s["test"]),
             n_train_COMIDs=s["train"].COMID.nunique(), n_test_COMIDs=s["test"].COMID.nunique())
        for specs, offset in [(publication_splits, 0), (fixed_splits, 9000)]
        for i, s in enumerate(specs, 1)])
    save_csv(split_audit, "split_audit.csv")
    (protocol_daily_predictions, protocol_event_predictions, protocol_event_daily_predictions,
     protocol_event_references, protocol_tuning, inner_membership) = fit_evaluation(
         publication_splits, daily_clim, reference_events)
    (fixed_daily_predictions, fixed_event_predictions, fixed_event_daily_predictions,
     fixed_event_references, fixed_tuning, fixed_inner_membership) = fit_evaluation(
         fixed_splits, daily_clim, reference_events, fixed=True)
    fixed_event_trend_long = make_event_trend_long(fixed_event_predictions)
    for name, table in {
        "event_window_daily_out_of_sample_predictions.csv": protocol_event_daily_predictions,
        "independent_event_references_by_split.csv": protocol_event_references,
        "fixed_hindcast_daily_predictions.csv": fixed_daily_predictions,
        "fixed_hindcast_event_predictions.csv": fixed_event_predictions,
        "fixed_hindcast_event_window_daily_predictions.csv": fixed_event_daily_predictions,
        "fixed_hindcast_event_references.csv": fixed_event_references,
        "inner_fold_membership.csv": pd.concat([inner_membership, fixed_inner_membership]),
    }.items():
        save_csv(table, name)

if "persistence_days" not in protocol_event_predictions:
    raise ValueError("Event cache lacks persistence_days; verify its provenance before replay.")
actual_persistence = set(protocol_event_predictions.persistence_days.dropna().astype(int))
if actual_persistence != {EVENT_PERSISTENCE_DAYS}:
    raise ValueError(f"Cached persistence {actual_persistence} differs from setting {EVENT_PERSISTENCE_DAYS}.")
for name, table in {
    "daily_out_of_sample_predictions.csv": protocol_daily_predictions,
    "independent_event_predictions.csv": protocol_event_predictions,
    "nested_tuning_by_protocol_split.csv": protocol_tuning,
    "fixed_hindcast_tuning_by_split.csv": fixed_tuning,
    "fixed_hindcast_phenology_event_trend_long.csv": fixed_event_trend_long,
}.items():
    save_csv(table, name)
print("Primary event records:", len(protocol_event_predictions))
print("Fixed trend-long rows:", len(fixed_event_trend_long), "(includes repeated series, not unique events)")

## 6 Daily ice state and event accuracy

Daily metrics are observation weighted. Event detection is reported on each model's available reference events; common complete-case metrics use identical records across **all eight model specifications**, preserving the original event-scoring convention. The primary paper table selects Full RF, Temperature LR, and the isotherm from these comparisons. Feature-ablation tables retain the six RF variants. MAE and RMSE are nonnegative; signed bias is predicted minus observed date.

In [ ]:
def metric_set(y, p, threshold=PROTOCOL_THRESHOLD, deterministic=False):
    y, p = np.asarray(y, int), np.asarray(p, float)
    if not np.isfinite(p).all():
        raise ValueError("Nonfinite daily probabilities found.")
    return dict(accuracy=accuracy_score(y, p >= threshold),
                brier=brier_score_loss(y, p),
                roc_auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                log_loss=np.nan if deterministic else log_loss(y, np.clip(p, 1e-12, 1 - 1e-12), labels=[0, 1]))

def summarize_daily(predictions):
    rows = []
    for evaluation, group in predictions.groupby("evaluation"):
        for model, col in MODEL_SPECS.items():
            scores = metric_set(group.is_ice, group[col], deterministic=model == "0 C isotherm")
            rows.extend(dict(evaluation=evaluation, model=model, weighting="observation", metric=m,
                             value=v, n=len(group), n_COMIDs=group.COMID.nunique()) for m, v in scores.items())
    return pd.DataFrame(rows)

def summarize_protocol_events(events):
    rows = []
    for (evaluation, event_type), g in events.groupby(["evaluation", "event_type"]):
        for model in MODEL_SPECS:
            paired = g.dropna(subset=["observed_date", model]).copy()
            errors = (paired[model] - paired.observed_date).dt.days.to_numpy()
            rows.append({
                "evaluation": evaluation, "event_type": event_type, "model": model,
                "n_observed_events": len(g), "n_predicted_events": len(paired),
                "event_detection_pct": 100 * len(paired) / len(g) if len(g) else np.nan,
                "MAE_days": np.mean(np.abs(errors)) if len(errors) else np.nan,
                "bias_days": np.mean(errors) if len(errors) else np.nan,
                "RMSE_days": np.sqrt(np.mean(errors ** 2)) if len(errors) else np.nan,
            })
    return pd.DataFrame(rows)

def summarize_events_common_complete_case(events, models=None):
    if models is None:
        models = list(MODEL_SPECS)

    rows = []
    required_cols = ["observed_date", *models]

    for (evaluation, event_type), g in events.groupby(["evaluation", "event_type"]):
        common = g.dropna(subset=required_cols).copy()

        for model in models:
            errors = (common[model] - common["observed_date"]).dt.days.to_numpy()
            rows.append({
                "evaluation": evaluation,
                "event_type": event_type,
                "model": model,
                "n_common_events": len(common),
                "n_common_COMIDs": common["COMID"].nunique(),
                "MAE_days": np.mean(np.abs(errors)) if len(errors) else np.nan,
                "bias_days": np.mean(errors) if len(errors) else np.nan,
                "RMSE_days": np.sqrt(np.mean(errors ** 2)) if len(errors) else np.nan,
            })

    return pd.DataFrame(rows)

def bootstrap_common_event_mae(events, models=None, n_boot=N_BOOT):
    models = list(MODEL_SPECS) if models is None else list(models)
    rows = []
    for (evaluation, outcome), group in events.groupby(["evaluation", "event_type"]):
        common = group.dropna(subset=["observed_date", *models]).copy()
        if common.empty:
            continue
        errors = pd.DataFrame({m: (common[m] - common.observed_date).dt.days.abs() for m in models})
        errors["COMID"] = common.COMID
        sums = errors.groupby("COMID", sort=False)[models].sum()
        counts = errors.groupby("COMID", sort=False).size().reindex(sums.index).to_numpy()
        for a, b in [("Random forest", m) for m in PRIMARY_MODELS if m != "Random forest"]:
            key = f"common_event|{evaluation}|{outcome}|{a}|{b}"
            rng = np.random.default_rng(SEED + zlib.crc32(key.encode()))
            draws = rng.integers(0, len(sums), size=(n_boot, len(sums)))
            delta = (sums[a] - sums[b]).to_numpy()
            boot = delta[draws].sum(axis=1) / counts[draws].sum(axis=1)
            lo, hi = np.quantile(boot, [0.025, 0.975])
            rows.append(dict(evaluation=evaluation, outcome=outcome, model_a=a, model_b=b,
                metric="MAE_days_common_complete_case", difference_a_minus_b=delta.sum()/counts.sum(),
                **{"CI_2.5%":lo, "CI_97.5%":hi}, better_direction="lower",
                n_common_events=len(common), n_common_COMIDs=len(sums), n_boot_valid=n_boot, cluster="COMID"))
    return pd.DataFrame(rows)

protocol_ice_metrics = summarize_daily(protocol_daily_predictions)
protocol_event_metrics = summarize_protocol_events(protocol_event_predictions)
protocol_event_metrics_common = summarize_events_common_complete_case(protocol_event_predictions)
protocol_event_pairwise_ci_common = bootstrap_common_event_mae(protocol_event_predictions)
freezeup_ablation_summary = protocol_event_metrics_common.query("event_type == 'freezeup'").copy()
freezeup_ablation_summary = freezeup_ablation_summary[freezeup_ablation_summary.model.isin(MODEL_FEATURE_SETS)]
for name, table in {
    "ice_presence_metrics.csv": protocol_ice_metrics,
    "breakup_freezeup_metrics.csv": protocol_event_metrics,
    "breakup_freezeup_metrics_common_complete_case.csv": protocol_event_metrics_common,
    "event_MAE_pairwise_COMID_bootstrap_CI_common_complete_case.csv": protocol_event_pairwise_ci_common,
    "freezeup_feature_ablation_summary.csv": freezeup_ablation_summary,
}.items():
    save_csv(table, name)
display(protocol_ice_metrics.pivot(index=["evaluation", "metric"], columns="model", values="value").round(4))
display(protocol_event_metrics_common.round(3))
display(protocol_event_pairwise_ci_common.round(3))

## 7 Event persistence sensitivity and validation audits

Set `RUN_PERSISTENCE_SENSITIVITY=True` to regenerate the 1/3/5-day comparison from continuous event-window predictions and reference events. This does not refit classifiers.

The deterministic-isotherm audit compares only shared events; two missing predictions count as the same detection status. The temporal-boundary audit identifies potentially overlapping autumn periods and is exported.

In [ ]:
def extract_persistence_sensitivity(event_daily_predictions, event_references, persistence_values=(1, 3, 5)):
    parts = []
    ref_groups = {sid: g.copy() for sid, g in event_references.groupby("split_id", sort=False)}

    for split_id, daily_split in event_daily_predictions.groupby("split_id", sort=False):
        refs = ref_groups.get(split_id)
        if refs is None or refs.empty:
            continue

        for persistence_days in persistence_values:
            events = extract_event_predictions(
                daily_split,
                refs,
                persistence_days=persistence_days,
            )
            if events.empty:
                continue

            events["evaluation"] = daily_split["evaluation"].iloc[0]
            events["split_id"] = split_id
            parts.append(events)

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

if RUN_PERSISTENCE_SENSITIVITY:
    if MODE == "reuse_predictions":
        protocol_event_daily_predictions = read_cache("event_window_daily_out_of_sample_predictions.csv", ["Date"])
        protocol_event_references = read_cache("independent_event_references_by_split.csv", ["observed_date"])
    persistence_events = extract_persistence_sensitivity(protocol_event_daily_predictions,
                                                        protocol_event_references, PERSISTENCE_VALUES)
    parts = []
    for days, group in persistence_events.groupby("persistence_days"):
        summary = summarize_protocol_events(group)
        summary["persistence_days"] = days
        parts.append(summary)
    persistence_metrics = pd.concat(parts, ignore_index=True)
    save_csv(persistence_metrics, "breakup_freezeup_metrics_persistence_sensitivity.csv")
    display(persistence_metrics.round(3))
else:
    print("Persistence sensitivity not executed; enable it if reporting this analysis.")

audit = protocol_event_predictions.groupby(["evaluation", "event_type"]).agg(
    n_reference_events=("COMID", "size"), n_COMIDs=("COMID", "nunique"),
    n_years=("WinterYear", "nunique")).reset_index()
save_csv(audit, "event_cohort_audit.csv")
boundary_parts = []
for origin in FORECAST_ORIGINS:
    group = protocol_event_predictions.loc[
        protocol_event_predictions.evaluation.isin(["time", "space_time"])
        & protocol_event_predictions.WinterYear.eq(origin["test_start"])
        & protocol_event_predictions.event_type.eq("freezeup")].copy()
    if not group.empty:
        group["potential_training_autumn_overlap"] = True
        boundary_parts.append(group[["evaluation", "split_id", "COMID", "WinterYear", "observed_date",
                                     "potential_training_autumn_overlap"]])
boundary_audit = (pd.concat(boundary_parts, ignore_index=True) if boundary_parts else
                  pd.DataFrame(columns=["evaluation", "split_id", "COMID", "WinterYear", "observed_date",
                                        "potential_training_autumn_overlap"]))
save_csv(boundary_audit, "first_test_winter_freezeup_boundary_audit.csv")
if not boundary_audit.empty:
    warnings.warn(f"{len(boundary_audit)} first-test-winter freeze-up records may overlap training autumns; inspect boundary audit.")

deterministic_rows = []
for a, b in combinations(sorted(protocol_event_predictions.evaluation.unique()), 2):
    keys = ["COMID", "WinterYear", "event_type"]
    left = protocol_event_predictions.query("evaluation == @a")[keys + ["0 C isotherm"]]
    right = protocol_event_predictions.query("evaluation == @b")[keys + ["0 C isotherm"]]
    shared = left.merge(right, on=keys, suffixes=("_a", "_b"), validate="one_to_one")
    same = (shared["0 C isotherm_a"].eq(shared["0 C isotherm_b"]) |
            (shared["0 C isotherm_a"].isna() & shared["0 C isotherm_b"].isna()))
    deterministic_rows.append(dict(evaluation_a=a, evaluation_b=b, n_shared=len(shared),
                                   n_disagree=int((~same).sum())))
save_csv(pd.DataFrame(deterministic_rows), "isotherm_shared_event_audit.csv")
display(audit)

## 8 Unified matched site-level trend analysis

All three primary models and observations use the same complete annual event records at a site. BH correction is applied across eligible sites separately within each event type and series. These MK tests do not correct for serial correlation, and BH control depends on the dependence structure among site tests.

Sites receive equal weight. Mean and median slopes and paired model-minus-observed summaries are exported separately; `TREND_SUMMARY` selects the paper-facing table. Bootstrap intervals resample sites with the four series retained together/.

In [ ]:
def mk_p_value(y):
    """Two-sided Mann–Kendall: tie-corrected variance, continuity correction."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    i, j = np.triu_indices(n, k=1)
    statistic = np.sign(y[j] - y[i]).sum()
    _, ties = np.unique(y, return_counts=True)
    variance = (
        n * (n - 1) * (2 * n + 5)
        - np.sum(ties * (ties - 1) * (2 * ties + 5))
    ) / 18.0
    z = (statistic - np.sign(statistic)) / np.sqrt(variance) if variance > 0 else 0.0
    return float(2 * norm.sf(abs(z)))

def bh_adjust(p):
    """BH-adjusted p-values for one event/series across eligible sites."""
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adjusted = np.minimum.accumulate(
        (p[order] * len(p) / np.arange(1, len(p) + 1))[::-1]
    )[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.minimum(adjusted, 1.0)
    return result

def matched_site_trends(event_long):
    data = event_long.loc[event_long.evaluation.eq("fixed_space_time") & event_long.series.isin(TREND_SERIES)].copy()
    keys = ["event_type", "COMID", "WinterYear", "series"]
    if data.empty or data[keys].isna().any().any() or data.duplicated(keys).any():
        raise ValueError("Missing/duplicated fixed-space-time trend identifiers.")
    for column in ["WinterYear", "phenology_day"]:
        data[column] = pd.to_numeric(data[column], errors="raise")
    if not data.WinterYear.between(2000, 2021).all():
        raise ValueError("Unexpected fixed-hindcast years.")
    rows, audit = [], []
    for event, group in data.groupby("event_type", sort=True):
        wide = group.pivot(index=["COMID", "WinterYear"], columns="series", values="phenology_day")
        if not set(TREND_SERIES).issubset(wide):
            raise ValueError(f"Missing primary trend series for {event}.")
        wide = wide[TREND_SERIES].sort_index()
        complete = wide.loc[np.isfinite(wide.to_numpy(dtype=float)).all(axis=1)]
        for comid, all_years in wide.groupby(level="COMID", sort=True):
            g = complete.loc[complete.index.get_level_values("COMID") == comid]
            audit.append(dict(event_type=event, COMID=comid, n_years_before_matching=len(all_years),
                              n_matched_years=len(g), included=len(g) >= MIN_TREND_YEARS))
            if len(g) < MIN_TREND_YEARS:
                continue
            x = g.index.get_level_values("WinterYear").to_numpy(float)
            for series in TREND_SERIES:
                y = g[series].to_numpy(float)
                rows.append(dict(evaluation="fixed_space_time", event_type=event, COMID=comid, series=series,
                    n_years=len(g), first_year=int(x.min()), last_year=int(x.max()),
                    slope_days_per_decade=10 * float(theilslopes(y, x).slope), mk_p=mk_p_value(y)))
    trends = pd.DataFrame(rows)
    trends["mk_significant"] = trends.mk_p < TREND_ALPHA
    trends["mk_fdr_p"] = trends.groupby(["event_type", "series"])["mk_p"].transform(bh_adjust)
    trends["mk_fdr_significant"] = trends.mk_fdr_p < TREND_ALPHA
    return trends, pd.DataFrame(audit)

def summarize_site_trends(trends, n_boot=TREND_N_BOOT):
    summaries, paired, sensitivity = [], [], []
    rng = np.random.default_rng(SEED)
    for event, group in trends.groupby("event_type", sort=True):
        wide = group.pivot(index="COMID", columns="series", values="slope_days_per_decade")[TREND_SERIES].sort_index()
        values = wide.to_numpy(float)
        draws = rng.integers(0, len(wide), size=(n_boot, len(wide)))
        for method in ["mean", "median"]:
            reducer = np.mean if method == "mean" else np.median
            boot = reducer(values[draws], axis=1)
            for j, series in enumerate(TREND_SERIES):
                lo, hi = np.quantile(boot[:, j], [.025, .975])
                summaries.append(dict(event_type=event, series=series, summary=method, n_sites=len(wide),
                    slope_days_per_decade=reducer(values[:, j]), **{"CI_2.5%":lo,"CI_97.5%":hi}))
            for a, b in [(m, "Observed") for m in PRIMARY_MODELS] + list(combinations(PRIMARY_MODELS, 2)):
                delta = (wide[a] - wide[b]).to_numpy()
                lo, hi = np.quantile(reducer(delta[draws], axis=1), [.025, .975])
                paired.append(dict(event_type=event, series_a=a, series_b=b, summary=method,
                    n_sites=len(wide), paired_slope_difference_days_per_decade=reducer(delta),
                    **{"CI_2.5%":lo,"CI_97.5%":hi}, n_boot=n_boot))
            if event == "breakup":
                counts = group.drop_duplicates("COMID").set_index("COMID").n_years
                for threshold in RECORD_THRESHOLDS:
                    selected = wide.loc[counts.reindex(wide.index).ge(threshold)]
                    record = dict(summary=method, min_observations=threshold, n_sites=len(selected))
                    record.update({s: reducer(selected[s].to_numpy()) if len(selected) else np.nan for s in TREND_SERIES})
                    sensitivity.append(record)
    return pd.DataFrame(summaries), pd.DataFrame(paired), pd.DataFrame(sensitivity)

site_level_trends, trend_inclusion_audit = matched_site_trends(fixed_event_trend_long)
trend_summaries, paired_site_trend_ci, threshold_sensitivity = summarize_site_trends(site_level_trends)
trend_significance_counts = site_level_trends.groupby(["event_type", "series"]).agg(
    n_sites=("COMID", "nunique"), n_MK_significant=("mk_significant", "sum"),
    n_FDR_significant=("mk_fdr_significant", "sum")).reset_index()
for name, table in {
    "matched_site_sen_slopes_and_significance.csv":site_level_trends,
    "trend_site_inclusion_audit.csv":trend_inclusion_audit,
    "site_trend_mean_and_median_CI.csv":trend_summaries,
    "paired_site_sen_slope_difference_CI.csv":paired_site_trend_ci,
    "trend_significance_counts.csv":trend_significance_counts,
    "site_trend_record_length_sensitivity.csv":threshold_sensitivity,
}.items(): save_csv(table, name)
display(trend_summaries.query("summary == @TREND_SUMMARY").round(3))
display(paired_site_trend_ci.query("summary == @TREND_SUMMARY and series_b == 'Observed'").round(3))
display(threshold_sensitivity.query("summary == @TREND_SUMMARY").round(3))
display(trend_significance_counts)

## 9 Paper figures and supplementary tables

Figure 3 uses the primary models scored on the common all-model event cohort. Figure 4 uses the matched ≥5-year break-up sites and unadjusted MK categories. Its supplied display limits are configurable; any points outside the plotted range are counted on the panel. Per-site Sen slopes do not change when their across-site summary changes from mean to median.

In [ ]:
def plot_event_errors(events):
    rng = np.random.default_rng(SEED)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
    for ax, outcome, title in zip(axes, ["breakup", "freezeup"], ["Break-up", "Freeze-up"]):
        group = events.query("evaluation == 'space_time' and event_type == @outcome").dropna(subset=["observed_date", *MODEL_SPECS])
        if group.empty:
            raise ValueError(f"No common events for Figure 3 {outcome}.")
        arrays = [(group[m] - group.observed_date).dt.days.to_numpy() for m in PRIMARY_MODELS]
        parts = ax.violinplot(arrays, positions=np.arange(3), showextrema=False)
        for body in parts["bodies"]:
            body.set_facecolor("0.75"); body.set_edgecolor("0.35"); body.set_alpha(.7)
        for j, errors in enumerate(arrays):
            ax.scatter(j + rng.uniform(-.10, .10, len(errors)), errors, s=5, color="black", alpha=.12, zorder=3)
            ax.text(j, 1.02, f"MAE: {np.abs(errors).mean():.1f} days\nRMSE: {np.sqrt(np.mean(errors**2)):.1f} days",
                    transform=ax.get_xaxis_transform(), ha="center", va="bottom", fontsize=9, color="black")
        ax.axhline(0, color="black", linestyle="--", lw=1)
        ax.set_axisbelow(True); ax.grid(axis="y", color=".88")
        ax.set_xticks(range(3), [MODEL_LABELS[m] for m in PRIMARY_MODELS], rotation=25, ha="right")
        ax.set_title(title, color="black", fontweight="bold", pad=45)
        ax.tick_params(colors="black")
    axes[0].set_ylabel("Predicted − observed event date (days)", color="black")
    fig.subplots_adjust(top=.73, bottom=.22, wspace=.10)
    fig.savefig(OUTPUT_DIR / "figure3_event_errors.png", dpi=600, bbox_inches="tight")
    plt.show()

def plot_site_trends(trends):
    group = trends.query("event_type == 'breakup'")
    obs = group.query("series == 'Observed'")[["COMID", "slope_days_per_decade", "mk_significant"]].rename(
        columns={"slope_days_per_decade":"observed_slope", "mk_significant":"observed_significant"})
    paired = group[group.series.isin(PRIMARY_MODELS)].merge(obs, on="COMID", validate="many_to_one")
    paired["category"] = np.select([paired.mk_significant & paired.observed_significant,
        ~paired.mk_significant & paired.observed_significant, paired.mk_significant & ~paired.observed_significant],
        ["Both significant", "Observed only", "Model only"], default="Neither significant")
    styles = {"Both significant":("black","D",.95,55), "Observed only":("#7B3294","^",.85,60),
              "Model only":("#009E73","s",.85,50), "Neither significant":(".70","o",.32,35)}
    values = paired[["observed_slope", "slope_days_per_decade"]].to_numpy()
    limits = FIGURE4_LIMITS if FIGURE4_LIMITS is not None else (5*np.floor(values.min()/5)-5, 5*np.ceil(values.max()/5)+5)
    fig, axes = plt.subplots(3, 1, figsize=(5.6,8.2), sharex=True, sharey=True)
    for ax, model in zip(axes, PRIMARY_MODELS):
        d = paired[paired.series.eq(model)]
        ax.set_axisbelow(True); ax.grid(color=".88", lw=.7)
        ax.plot(limits, limits, "k--", lw=1, zorder=1)
        ax.axhline(0,color=".72",lw=.8,zorder=1); ax.axvline(0,color=".72",lw=.8,zorder=1)
        for name in ["Neither significant", "Observed only", "Model only", "Both significant"]:
            color, marker, alpha, size = styles[name]
            dd = d[d.category.eq(name)]
            ax.scatter(dd.observed_slope,dd.slope_days_per_decade,color=color,marker=marker,alpha=alpha,s=size,
                       edgecolors="none" if name == "Neither significant" else "black",linewidths=.6,zorder=3)
        ax.set_title(MODEL_LABELS[model],fontweight="bold",color="black",pad=6)
        ax.set_xlim(limits);ax.set_ylim(limits);ax.tick_params(colors="black")
        outside=(~d.observed_slope.between(*limits)|~d.slope_days_per_decade.between(*limits)).sum()
        if outside: ax.text(.02,.96,f"{outside} sites outside displayed limits",transform=ax.transAxes,va="top",fontsize=8)
    handles=[Line2D([0],[0],linestyle="none",marker=m,markerfacecolor=c,
                   markeredgecolor="none" if name == "Neither significant" else "black",markersize=7,alpha=a)
             for name,(c,m,a,s) in styles.items()]
    fig.suptitle("Site-level Mann–Kendall break-up trends",y=.985,fontweight="bold",color="black")
    fig.legend(handles,list(styles),loc="upper center",bbox_to_anchor=(.5,.94),ncol=2,frameon=False)
    fig.supxlabel("Observed Sen slope (days decade$^{-1}$)",y=.025,color="black")
    fig.supylabel("Modeled Sen slope (days decade$^{-1}$)",x=.035,color="black")
    fig.subplots_adjust(left=.17,right=.98,bottom=.085,top=.82,hspace=.34)
    fig.savefig(OUTPUT_DIR / "figure4_fixed_matched_stacked.png",dpi=600,bbox_inches="tight")
    save_csv(paired,"figure4_fixed_matched_plot_data.csv")
    plt.show()

plot_event_errors(protocol_event_predictions)
plot_site_trends(site_level_trends)

# Export clearly named, paper-facing tables.
save_csv(protocol_ice_metrics[protocol_ice_metrics.model.isin(PRIMARY_MODELS)], "main_daily_metrics.csv")
save_csv(protocol_event_metrics_common[protocol_event_metrics_common.model.isin(PRIMARY_MODELS)], "main_event_metrics.csv")
save_csv(threshold_sensitivity.query("summary == @TREND_SUMMARY"), "supplement_record_length_sensitivity.csv")
save_csv(pd.DataFrame([dict(model=m, features=", ".join(f)) for m,f in MODEL_FEATURE_SETS.items()]), "supplement_feature_sets.csv")
tuning_records=[]
for table, offset in [(protocol_tuning,0),(fixed_tuning,9000)]:
    for number, row in enumerate(table.itertuples(index=False),1):
        params, scores = json.loads(row.rf_best_params), json.loads(row.rf_inner_brier)
        for model, selected in params.items():
            tuning_records.append(dict(evaluation=row.evaluation,split_id=row.split_id,model=model,
                seed=getattr(row,"seed",SEED+offset+number*100),inner_brier=scores[model],**selected))
save_csv(pd.DataFrame(tuning_records),"selected_parameters_all_outer_splits.csv")

## 10 Archive metadata

In [ ]:
metadata = {
    "source_notebook": "Final_RF_vs_TempRF_publication_evaluation_v4_MK_site_trends(1).ipynb",
    "mode":MODE, "seed":SEED, "event_persistence_days":EVENT_PERSISTENCE_DAYS,
    "probability_threshold":PROTOCOL_THRESHOLD, "inner_folds":N_INNER_SPLITS,
    "outer_spatial_folds":N_OUTER_SPLITS, "sampled_candidates":N_SEARCH_ITER,
    "forecast_origins":FORECAST_ORIGINS, "fixed_training_end":1999,
    "fixed_test_years":[2000,2021], "event_bootstrap_replicates":N_BOOT,
    "trend_bootstrap_replicates":TREND_N_BOOT, "trend_summary":TREND_SUMMARY,
    "trend_min_matched_years":MIN_TREND_YEARS, "trend_models":TREND_SERIES,
    "FDR_family":"eligible sites separately within each event type and series",
    "event_common_cohort_models":list(MODEL_SPECS), "feature_sets":MODEL_FEATURE_SETS,
    "rf_search_space":rf_param_distributions, "software_versions":software_versions,
    "persistence_sensitivity_executed":RUN_PERSISTENCE_SENSITIVITY,
    "potential_boundary_overlap_records":len(boundary_audit),
    "fixed_trend_persistence_provenance":"Verify matching archived protocol; the long CSV has no persistence column.",
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
(OUTPUT_DIR / "input_manifest.json").write_text(json.dumps(input_manifest,indent=2),encoding="utf-8")
requirements = [f"{name}=={version}" for name,version in software_versions.items() if name != "python"]
(OUTPUT_DIR / "requirements_executed.txt").write_text("\n".join(requirements)+"\n",encoding="utf-8")
print("Publication outputs:",OUTPUT_DIR.resolve())
print("Archive the notebook, inputs/predictions, metadata, figures, and final executed environment together.")